# PR0204. Pipeline de ingesta y procesamiento en HDFS

**Alumno: Diego Blanco Barredo**

Cargo tres CSV de Renfe en HDFS, filtro las estaciones de León y descargo
el resultado. Ejecuto el cuaderno en el Jupyter del contenedor `namenode`
con la librería `hdfs`.

Los CSV originales van en `datos/`. Si faltan, el cuaderno los descarga
del portal oficial. Los he renombrado para seguir el enunciado.

Datos de Renfe, con licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

- `estaciones_completo.csv`. [Estaciones. Listado completo](https://data.renfe.com/dataset/estaciones-listado-completo)
- `estaciones_feve.csv`. [Estaciones de FEVE](https://data.renfe.com/dataset/listado-estaciones-feve)
- `viajeros_tramo.csv`. [Viajeros por franja horaria de León AM](https://data.renfe.com/dataset/volumen-de-viajeros-por-franja-horaria-leon-am)

In [1]:
import csv
import io
import os
import shutil
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from itertools import islice
from pathlib import Path
from urllib.request import urlopen

from hdfs import InsecureClient

WEBHDFS = os.environ.get("PR0204_WEBHDFS", "http://namenode:9870")
PROVINCIA = "LEÓN"
HILOS = 4
RAIZ = "/datalake/renfe"
RAW = f"{RAIZ}/raw"
RAW_ESTACIONES = f"{RAW}/estaciones"
RAW_VIAJEROS = f"{RAW}/viajeros"
PROCESSED = f"{RAIZ}/processed"
LOGS = f"{RAIZ}/logs"

ESTACIONES = f"{RAW_ESTACIONES}/estaciones_completo.csv"
FEVE = f"{RAW_ESTACIONES}/estaciones_feve.csv"
VIAJEROS = f"{RAW_VIAJEROS}/viajeros_tramo.csv"
FILTRADAS = f"{PROCESSED}/estaciones_filtradas.csv"
REGISTRO = f"{LOGS}/ingesta.log"
ENTREGA = Path("entrega_estaciones.csv")

# Enlaces de descarga de las fichas oficiales de Renfe.
fuentes = {
    ESTACIONES: "https://ssl.renfe.com/ftransit/Fichero_estaciones/estaciones.csv",
    FEVE: (
        "https://data.renfe.com/dataset/f72abe7d-fea8-4af1-9e3d-cf0997b81955/"
        "resource/4de015e6-987c-43ae-9350-8bf941ef66c7/download/listado-de-estaciones-feve-2.csv"
    ),
    VIAJEROS: (
        "https://data.renfe.com/dataset/fae11772-844a-4c0c-a158-0f0614a86490/"
        "resource/5a3c3597-4a03-4ce8-adae-df6f25da7a14/download/leon-am_viajeros_por_franja_csv.csv"
    ),
}

datos = Path("datos")
datos.mkdir(exist_ok=True)
archivos = {}
for ruta_hdfs, url in fuentes.items():
    ruta_local = datos / ruta_hdfs.rsplit("/", 1)[1]
    if not ruta_local.is_file():
        temporal = ruta_local.with_suffix(".part")
        with urlopen(url, timeout=60) as respuesta, temporal.open("wb") as salida:
            shutil.copyfileobj(respuesta, salida, length=65536)
        temporal.replace(ruta_local)
    if ruta_local.stat().st_size == 0:
        raise ValueError(f"El CSV está vacío: {ruta_local}")
    archivos[ruta_hdfs] = ruta_local

# La cabecera de Renfe tiene acentos, que permiten distinguir ambas codificaciones.
with archivos[ESTACIONES].open("rb") as archivo:
    cabecera_local = archivo.readline(8192)
try:
    cabecera_local.decode("utf-8-sig")
    CODIFICACION = "utf-8-sig"
except UnicodeDecodeError:
    CODIFICACION = "cp1252"
SEPARADOR = ";"

def normalizar(texto):
    letras = unicodedata.normalize("NFD", texto.strip().upper())
    return "".join(letra for letra in letras if not unicodedata.combining(letra))

campos_locales = next(csv.reader([cabecera_local.decode(CODIFICACION)], delimiter=SEPARADOR))
if "PROVINCIA" not in [normalizar(campo) for campo in campos_locales]:
    raise ValueError("El listado completo de Renfe debe contener la columna PROVINCIA.")

## 1. Conexión y estructura del Data Lake

Creo las carpetas y compruebo que tienen permisos `755`.

In [2]:
cliente = InsecureClient(WEBHDFS, user="root", timeout=(5, 60))
cliente.status("/")

carpetas = (RAIZ, RAW, RAW_ESTACIONES, RAW_VIAJEROS, PROCESSED, LOGS)
for ruta in carpetas:
    cliente.makedirs(ruta, permission="755")
    cliente.set_permission(ruta, "755")

for ruta in carpetas:
    estado = cliente.status(ruta)
    if estado["type"] != "DIRECTORY" or estado["permission"] != "755":
        raise RuntimeError(f"No se ha comprobado la carpeta {ruta} con permisos 755.")
    print(f"{ruta} | {estado['permission']}")
print("Contenido de /datalake/renfe:", ", ".join(sorted(cliente.list(RAIZ))))

/datalake/renfe | 755
/datalake/renfe/raw | 755
/datalake/renfe/raw/estaciones | 755
/datalake/renfe/raw/viajeros | 755
/datalake/renfe/processed | 755
/datalake/renfe/logs | 755
Contenido de /datalake/renfe: logs, processed, raw


## 2. Ingesta en la zona raw

Configuro cuatro hilos y permito sobrescribir los tres archivos al repetir
la práctica. Consulto los metadatos de las estaciones después de subirlas.

In [3]:
def subir_csv(archivo):
    ruta_hdfs, ruta_local = archivo
    conexion = InsecureClient(WEBHDFS, user="root", timeout=(5, 60))
    conexion.upload(ruta_hdfs, str(ruta_local), overwrite=True, n_threads=HILOS)

with ThreadPoolExecutor(max_workers=HILOS) as grupo:
    for _ in grupo.map(subir_csv, archivos.items()):
        pass

for ruta_hdfs, ruta_local in archivos.items():
    if cliente.status(ruta_hdfs)["length"] != ruta_local.stat().st_size:
        raise RuntimeError(f"El tamaño subido no coincide con el original: {ruta_hdfs}")

for nombre, estado in sorted(cliente.list(RAW_ESTACIONES, status=True)):
    if nombre in ("estaciones_completo.csv", "estaciones_feve.csv"):
        print(f"{nombre} | {estado['length']} | {estado['permission']} | {estado['replication']}")

estaciones_completo.csv | 142155 | 644 | 3
estaciones_feve.csv | 61795 | 644 | 3


## 3. Filtrado de estaciones en streaming

Leo el CSV de HDFS línea a línea y comparo el valor de `PROVINCIA` con
León. Conservo la cabecera y envío cada fila seleccionada directamente a
HDFS mediante un generador. El resultado se guarda en UTF-8.

In [4]:
def filtrar_estaciones():
    leidas = 0
    seleccionadas = 0

    def generar_filas():
        nonlocal leidas, seleccionadas
        with cliente.read(ESTACIONES, encoding=CODIFICACION, delimiter="\n") as lineas:
            cabecera = next(lineas).rstrip("\r\n")
            campos = next(csv.reader([cabecera], delimiter=SEPARADOR))
            indice = [normalizar(campo) for campo in campos].index("PROVINCIA")
            yield cabecera + "\n"

            lector = csv.reader((linea + "\n" for linea in lineas), delimiter=SEPARADOR)
            buffer_fila = io.StringIO()
            escritor = csv.writer(buffer_fila, delimiter=SEPARADOR, lineterminator="\r\n")
            for fila in lector:
                if not fila or not any(campo.strip() for campo in fila):
                    continue
                if len(fila) != len(campos):
                    raise ValueError(f"Número de columnas incorrecto cerca de la línea {lector.line_num + 1}.")
                leidas += 1
                if normalizar(fila[indice]) == normalizar(PROVINCIA):
                    buffer_fila.seek(0)
                    buffer_fila.truncate(0)
                    escritor.writerow(fila)
                    seleccionadas += 1
                    yield buffer_fila.getvalue()

    cliente.write(FILTRADAS, data=generar_filas(), encoding="utf-8", overwrite=True)
    return leidas, seleccionadas

total_leidas, total_filtradas = filtrar_estaciones()
print("Provincia:", PROVINCIA)
print("Codificación del original:", CODIFICACION)
print("Estaciones leídas:", total_leidas)
print("Estaciones filtradas:", total_filtradas)
print("Primeras cinco líneas del resultado:")
with cliente.read(FILTRADAS, encoding="utf-8", delimiter="\n") as lineas:
    for linea in islice(lineas, 5):
        print(linea.strip())

Provincia: LEÓN
Codificación del original: cp1252
Estaciones leídas: 1033
Estaciones filtradas: 43
Primeras cinco líneas del resultado:
"CODIGO";"DESCRIPCION";"LATITUD";"LONGITUD";"DIRECION";"CP";"POBLACION";"PROVINCIA";"PAIS";"CERCANIAS";"ANCHO MÉTRICO";"COMUN"
05755;LA LLAMA DE LA GUZPEÑA;42.7702481;-5.0412478;Ctra. Cl - 626 --- Estación Ferrocarril, S/N;24893;PRADO DE LA GUZPEÑA;LEÓN;ESPAÑA;SI;SI;
05756;LA ESPINA;42.7868560;-4.9052025;Ctra. Cl - 626 --- Estación Ferrocarril, S/N;24889;VALDERRUEDA;LEÓN;ESPAÑA;SI;SI;
05757;VALCUENDE;42.7655780;-4.9292558;C/ Arriba, S/N;24889;VALDERRUEDA;LEÓN;ESPAÑA;SI;SI;
05758;PUENTE ALMUHEY;42.7876320;-4.9718992;Ctra. General  S/N- Puente Almuhey;24880;VALDERRUEDA;LEÓN;ESPAÑA;SI;SI;


## 4. Registro de auditoría

Escribo el inicio de esta ejecución y añado el total filtrado al mismo log
con `append=True`.

In [5]:
inicio = "[INICIO] Pipeline ejecutado. Archivos crudos cargados en /raw.\n"
procesado = (
    f"[PROCESADO] Se han filtrado un total de {total_filtradas} "
    "estaciones para la provincia seleccionada.\n"
)

cliente.write(REGISTRO, data=(inicio,), overwrite=True, encoding="utf-8")
cliente.write(REGISTRO, data=(procesado,), append=True, encoding="utf-8")
with cliente.read(REGISTRO, encoding="utf-8") as archivo:
    log = archivo.read()
if log != inicio + procesado:
    raise RuntimeError("El registro de auditoría no contiene los dos eventos esperados.")
print(log.strip())

[INICIO] Pipeline ejecutado. Archivos crudos cargados en /raw.
[PROCESADO] Se han filtrado un total de 43 estaciones para la provincia seleccionada.


## 5. Descarga y limpieza

Descargo el CSV filtrado, compruebo el número de filas y elimino el CSV de
FEVE. El listado final muestra que el fichero de estaciones completo sigue
en HDFS.

In [6]:
cliente.download(FILTRADAS, str(ENTREGA), overwrite=True)
with ENTREGA.open(encoding="utf-8", newline="") as archivo:
    lector = csv.reader(archivo, delimiter=SEPARADOR)
    campos = next(lector)
    indice = [normalizar(campo) for campo in campos].index("PROVINCIA")
    total_exportadas = 0
    for fila in lector:
        if len(fila) != len(campos) or normalizar(fila[indice]) != normalizar(PROVINCIA):
            raise RuntimeError("La exportación contiene una fila que no coincide con el filtro.")
        total_exportadas += 1
if total_exportadas != total_filtradas:
    raise RuntimeError("La descarga no tiene el mismo número de estaciones que el fichero de HDFS.")

print(f"Archivo descargado: {ENTREGA.name} ({ENTREGA.stat().st_size} bytes)")
print("Estaciones en la copia local:", total_exportadas)
estado_completo = cliente.status(ESTACIONES)
if not cliente.delete(FEVE):
    raise RuntimeError("No se ha podido eliminar estaciones_feve.csv.")
restantes = sorted(cliente.list(RAW_ESTACIONES))
if "estaciones_feve.csv" in restantes or cliente.status(ESTACIONES) != estado_completo:
    raise RuntimeError("El listado final de estaciones no es el esperado.")

print("Archivo eliminado: estaciones_feve.csv")
print("Archivos que quedan en raw/estaciones:", ", ".join(restantes))

Archivo descargado: entrega_estaciones.csv (4932 bytes)
Estaciones en la copia local: 43
Archivo eliminado: estaciones_feve.csv
Archivos que quedan en raw/estaciones: estaciones_completo.csv


La función de reinicio queda al final con la llamada comentada, de modo que
la práctica conserva los resultados en HDFS.

In [7]:
def reset_datalake():
    """Borra de forma recursiva los directorios y archivos de este proyecto."""
    return cliente.delete(RAIZ, recursive=True)

# reset_datalake()